In [ ]:
import math
from collections.abc import Sequence
from dataclasses import dataclass
from typing import NamedTuple, Protocol, TypeAlias

import numpy as np
import pandas as pd
import pydantic

## Scheduling

In [2]:
class FlexibleTripInfo(NamedTuple):
    ozone: int
    dzone: int
    purp: str
    mode: str
    # TODO: Somehow combine this information with the tuple below without mixing states.
    earliest_departure: pydantic.PositiveFloat
    latest_departure: pydantic.PositiveFloat


class RefinedDepartureWindow(NamedTuple):
    """Trip departure time window after refinement."""

    earliest_departure: pydantic.NonNegativeFloat
    """The earliest feasible departure time."""
    latest_departure: pydantic.NonNegativeFloat
    """The latest feasible departure time."""


class SchedulingError(Exception):
    """Raised when scheduling fails due to infeasible constraints."""

    def __init__(self, message: str) -> None:
        super().__init__(message)

In [3]:
class SchedulingSuccess(NamedTuple):
    scheduled_departures: list[float]


class SchedulingFailure(NamedTuple):
    error: SchedulingError


SchedulingResult: TypeAlias = SchedulingSuccess | SchedulingFailure

In [4]:
class TravelTimeFn(Protocol):
    def __call__(
        self, ozone: int, dzone: int, mode: str, time: pydantic.PositiveFloat
    ) -> pydantic.PositiveFloat: ...


class DepartureWindowRefiner(Protocol):
    def __call__(
        self,
        trip_info: Sequence[FlexibleTripInfo],
        travel_time_fn: TravelTimeFn,
        min_act_duration: pydantic.PositiveFloat,
    ) -> list[RefinedDepartureWindow]: ...


class DepartureWindowSampler(Protocol):
    def __call__(
        self,
        trip_info: Sequence[FlexibleTripInfo],
        windows: Sequence[RefinedDepartureWindow],
        travel_time_fn: TravelTimeFn,
        min_act_duration: pydantic.PositiveFloat,
        rng: np.random.Generator,
    ) -> list[pydantic.PositiveFloat]: ...

In [5]:
def refine_departure_windows(
    trip_info: Sequence[FlexibleTripInfo],
    travel_time_fn: TravelTimeFn,
    min_act_duration: pydantic.PositiveFloat,
) -> list[RefinedDepartureWindow]:
    earliest_departures = _refine_earliest_departures(
        trip_info, travel_time_fn, min_act_duration
    )
    latest_departures = _refine_latest_departures(
        trip_info, travel_time_fn, min_act_duration, earliest_departures
    )

    return [
        RefinedDepartureWindow(
            earliest_departure=earliest_departure, latest_departure=latest_departure
        )
        for earliest_departure, latest_departure in zip(
            earliest_departures, latest_departures
        )
    ]


def _refine_earliest_departures(
    trip_info: Sequence[FlexibleTripInfo],
    travel_time_fn: TravelTimeFn,
    min_act_duration: pydantic.PositiveFloat,
) -> list[pydantic.PositiveFloat]:
    earliest_departures = []
    for index, info in enumerate(trip_info):
        if index == 0:
            curr_earliest_departure = info.earliest_departure
        else:
            prev_info = trip_info[index - 1]
            prev_earliest_departure = earliest_departures[index - 1]

            travel_time = travel_time_fn(
                prev_info.ozone,
                prev_info.dzone,
                prev_info.mode,
                prev_earliest_departure,
            )
            curr_earliest_arrival = prev_earliest_departure + travel_time

            # Ensure that the earliest feasible departure is no earlier than the start of the corresponding window.
            # This can happen with sparse trip chains and relatively short travel times.
            curr_earliest_departure = max(
                curr_earliest_arrival + min_act_duration, info.earliest_departure
            )
            if curr_earliest_departure > info.latest_departure:
                raise SchedulingError(
                    f"The earliest feasible departure of Trip {index} ({curr_earliest_departure}) is later than the end of the corresponding window ({info.latest_departure})."
                )

        earliest_departures.append(curr_earliest_departure)

    return earliest_departures


def _refine_latest_departures(
    trip_info: Sequence[FlexibleTripInfo],
    travel_time_fn: TravelTimeFn,
    min_act_duration: pydantic.PositiveFloat,
    earliest_departures: list[pydantic.PositiveFloat],
) -> list[pydantic.PositiveFloat]:
    latest_departures = []
    for index, info in reversed(list(enumerate(trip_info))):
        curr_earliest_departure = earliest_departures[index]
        if index == len(trip_info) - 1:
            curr_latest_departure = info.latest_departure
        else:
            next_latest_departure = latest_departures[-1]
            curr_latest_arrival = next_latest_departure - min_act_duration

            curr_latest_departure = _calculate_latest_departure(
                trip_info=info,
                target_arrival=curr_latest_arrival,
                earliest_departure=curr_earliest_departure,
                travel_time_fn=travel_time_fn,
            )

            # TODO: Why this clamping necessary?
            curr_latest_departure = min(curr_latest_departure, info.latest_departure)
            if curr_latest_departure < curr_earliest_departure:
                raise SchedulingError(
                    f"The latest feasible departure time of Trip {index} is earlier than the start of the corresponding window ({info.earliest_departure})."
                )

        latest_departures.append(curr_latest_departure)

    return list(reversed(latest_departures))


def _calculate_latest_departure(
    trip_info: FlexibleTripInfo,
    target_arrival: pydantic.PositiveFloat,
    earliest_departure: pydantic.PositiveFloat,
    travel_time_fn: TravelTimeFn,
) -> pydantic.PositiveFloat:
    earliest_departure, latest_departure = (
        earliest_departure,
        trip_info.latest_departure,
    )
    for _ in range(100):
        if math.isclose(earliest_departure, latest_departure):
            break

        mid_departure = (earliest_departure + latest_departure) * 0.5

        travel_time = travel_time_fn(
            trip_info.ozone, trip_info.dzone, trip_info.mode, mid_departure
        )
        if mid_departure + travel_time <= target_arrival:
            earliest_departure = mid_departure
        else:
            latest_departure = mid_departure

    # This is the latest known departure.
    return earliest_departure

In [6]:
def sample_departures(
    trip_info: Sequence[FlexibleTripInfo],
    windows: Sequence[RefinedDepartureWindow],
    travel_time_fn: TravelTimeFn,
    min_act_duration: pydantic.PositiveFloat,
    rng: np.random.Generator,
) -> list[pydantic.PositiveFloat]:
    scheduled_departures = []
    for index, window in enumerate(windows):
        if index == 0:
            curr_earliest_departure = window.earliest_departure
        else:
            prev_info = trip_info[index - 1]
            prev_scheduled_departure = scheduled_departures[index - 1]

            travel_time = travel_time_fn(
                prev_info.ozone,
                prev_info.dzone,
                prev_info.mode,
                prev_scheduled_departure,
            )
            curr_scheduled_arrival = prev_scheduled_departure + travel_time

            curr_earliest_departure = max(
                curr_scheduled_arrival + min_act_duration, window.earliest_departure
            )
            if curr_earliest_departure > window.latest_departure:
                raise SchedulingError(
                    f"The scheduled departure of Trip {index} ({curr_earliest_departure}) is later than the end of the corresponding window ({window.latest_departure})."
                )

        # TODO: Check for duplicate departure times.
        curr_scheduled_departure = rng.uniform(
            curr_earliest_departure, window.latest_departure
        )
        scheduled_departures.append(curr_scheduled_departure)

    return scheduled_departures

In [7]:
# TODO: Should we add a `schedule_or_raise` function?
# TODO: Add multiple schedule sampling.
# TODO: Add a convenience function for inspecting refined departure windows.
def schedule(
    trip_info: Sequence[FlexibleTripInfo],
    travel_time_fn: TravelTimeFn,
    min_act_duration: pydantic.PositiveFloat,
    refiner: DepartureWindowRefiner,
    sampler: DepartureWindowSampler,
    rng: np.random.Generator,
) -> SchedulingResult:
    try:
        refined_windows = refiner(
            trip_info, travel_time_fn=travel_time_fn, min_act_duration=min_act_duration
        )
        scheduled_departures = sampler(
            trip_info,
            windows=refined_windows,
            travel_time_fn=travel_time_fn,
            min_act_duration=min_act_duration,
            rng=rng,
        )

        return SchedulingSuccess(scheduled_departures=scheduled_departures)
    except SchedulingError as e:
        return SchedulingFailure(e)

## Parsing

In [8]:
@dataclass(frozen=True)
class FlexibleTripColumnSpec:
    pid: str
    ozone: str
    dzone: str
    purp: str
    mode: str
    earliest_departure: str
    latest_departure: str

In [9]:
def parse_flexible(
    trips: pd.DataFrame,
    spec: FlexibleTripColumnSpec,
    travel_time_fn: TravelTimeFn,
    min_act_duration: pydantic.PositiveFloat,
    refiner: DepartureWindowRefiner,
    sampler: DepartureWindowSampler,
    rng: np.random.Generator,
) -> tuple[dict[int, SchedulingSuccess], dict[int, SchedulingFailure]]:
    successes: dict[int, SchedulingSuccess] = {}
    failures: dict[int, SchedulingFailure] = {}

    for (
        # TODO: Validate the variable naming scheme.
        index,
        group,
    ) in trips.groupby(spec.pid):
        # TODO: Catch refinement, travel time calculation, and sampling errors.
        try:
            scheduling_result = _parse_flexible(
                trips=group,
                spec=spec,
                travel_time_fn=travel_time_fn,
                min_act_duration=min_act_duration,
                refiner=refiner,
                sampler=sampler,
                rng=rng,
            )
        except pydantic.ValidationError as e:
            print(e)
            failures[index] = scheduling_result

        if isinstance(scheduling_result, SchedulingSuccess):
            successes[index] = scheduling_result
        else:
            failures[index] = scheduling_result

    return successes, failures


def _parse_flexible(
    trips: pd.DataFrame,
    spec: FlexibleTripColumnSpec,
    travel_time_fn: TravelTimeFn,
    min_act_duration: pydantic.PositiveFloat,
    refiner: DepartureWindowRefiner,
    sampler: DepartureWindowSampler,
    rng: np.random.Generator,
):
    # TODO: Validate the trip column specification.
    # TODO: Sort the trips by earliest departure.

    trip_info = [
        FlexibleTripInfo(
            ozone=getattr(record, spec.ozone),
            dzone=getattr(record, spec.dzone),
            purp=getattr(record, spec.purp),
            mode=getattr(record, spec.mode),
            earliest_departure=getattr(record, spec.earliest_departure),
            latest_departure=getattr(record, spec.latest_departure),
        )
        for record in trips.itertuples(index=False)
    ]

    scheduling_result = schedule(
        trip_info,
        travel_time_fn=travel_time_fn,
        min_act_duration=min_act_duration,
        refiner=refiner,
        sampler=sampler,
        rng=rng,
    )

    return scheduling_result

In [10]:
import json

from examples.v1.travel_time import TravelTimeCalculator

trips = pd.read_csv("res/survey/long/trips.csv")

routing = np.load("res/travel_time/routing.npz")

with open("res/travel_time/zone_encoder.json") as f:
    zone_encoder = json.load(f)

travel_time_fn = TravelTimeCalculator(
    driving_matrix=routing["driving_matrix"],
    transit_matrix=routing["transit_matrix"],
    zonal_lengths=routing["zonal_lengths"],
    zone_encoder=zone_encoder,
).calculate

In [14]:
spec = FlexibleTripColumnSpec(
    pid="pid",
    ozone="ozone",
    dzone="dzone",
    purp="purp",
    mode="mode",
    earliest_departure="min_time",
    latest_departure="max_time",
)

scheduling_results = parse_flexible(
    trips,
    spec=spec,
    travel_time_fn=travel_time_fn,
    min_act_duration=0.5,
    refiner=refine_departure_windows,
    sampler=sample_departures,
    rng=np.random.default_rng(seed=0),
)
scheduling_results

1 validation error for TravelTimeCalculator.calculate
  Input should be a finite number [type=finite_number, input_value=np.float64(nan), input_type=float64]
    For further information visit https://errors.pydantic.dev/2.12/v/finite_number


({101: SchedulingSuccess(scheduled_departures=[9.910885058405063, 17.80936013827651, 20.312533285591222, 23.099165813171176]),
  102: SchedulingSuccess(scheduled_departures=[13.439810708511814, 19.73826672163233, 22.0648013429436]),
  103: SchedulingSuccess(scheduled_departures=[13.188489674799236, 15.757272013601522]),
  104: SchedulingSuccess(scheduled_departures=[13.805217260913055, 16.630263705025026, 20.008215500510445]),
  105: SchedulingSuccess(scheduled_departures=[19.572212820180468, 20.54377102082841]),
  106: SchedulingSuccess(scheduled_departures=[13.188966331135298, 14.578781209150405]),
  107: SchedulingSuccess(scheduled_departures=[9.737866886021884, 10.873687947326395, 14.899135671612154]),
  108: SchedulingSuccess(scheduled_departures=[15.268061658869078, 17.08495901343639]),
  109: SchedulingSuccess(scheduled_departures=[17.372849828109718, 22.011873244080892]),
  110: SchedulingSuccess(scheduled_departures=[3.2359475548575394, 21.84615533444376]),
  111: SchedulingSu

In [15]:
scheduling_successes = scheduling_results[0]
scheduling_successes

{101: SchedulingSuccess(scheduled_departures=[9.910885058405063, 17.80936013827651, 20.312533285591222, 23.099165813171176]),
 102: SchedulingSuccess(scheduled_departures=[13.439810708511814, 19.73826672163233, 22.0648013429436]),
 103: SchedulingSuccess(scheduled_departures=[13.188489674799236, 15.757272013601522]),
 104: SchedulingSuccess(scheduled_departures=[13.805217260913055, 16.630263705025026, 20.008215500510445]),
 105: SchedulingSuccess(scheduled_departures=[19.572212820180468, 20.54377102082841]),
 106: SchedulingSuccess(scheduled_departures=[13.188966331135298, 14.578781209150405]),
 107: SchedulingSuccess(scheduled_departures=[9.737866886021884, 10.873687947326395, 14.899135671612154]),
 108: SchedulingSuccess(scheduled_departures=[15.268061658869078, 17.08495901343639]),
 109: SchedulingSuccess(scheduled_departures=[17.372849828109718, 22.011873244080892]),
 110: SchedulingSuccess(scheduled_departures=[3.2359475548575394, 21.84615533444376]),
 111: SchedulingSuccess(sched